# Deep-learning models demo (needs `uv sync --extra dl`)
Strategy #9 (1-D CNN / MLP / LSTM on a year of scaled returns) and #10 (conditional autoencoder). Both are early-stopped on a *later* slice of the training window. Synthetic data: the point is the pipeline, not the numbers. Not run in CI (core install has no PyTorch).

In [1]:
import pandas as pd

from quantskills.data import synthetic_prices
from quantskills.eval import tear_sheet
from quantskills.models.nn import torch_available
from quantskills.strategies import ConditionalAutoencoder, DLForecast, run_strategy
from quantskills.utils import holdout_date

assert torch_available(), "run: uv sync --extra dl"
prices = synthetic_prices(seed=0)
strong = synthetic_prices(seed=0, drift_vol=0.0012)  # positive control: 3x drift
train_end = holdout_date(prices.index.get_level_values("date"), 0.3)

## #9: does the network learn anything?
Watch the validation IC and the epoch early stopping picks. On the default panel it stops almost at once; with 3x stronger drift the CNN learns.

In [2]:
rows = []
for label, px in (("default", prices), ("3x drift", strong)):
    for arch in ("mlp", "cnn", "lstm"):
        s = DLForecast(arch=arch)
        run = run_strategy(s, px, train_end)
        rows.append(
            {
                "panel": label,
                "arch": arch,
                "best_epoch": s.train_report_.best_epoch,
                "val_IC": s.train_report_.val_score,
                "IS_sharpe": tear_sheet(run.in_sample)["sharpe"],
                "holdout_sharpe": tear_sheet(run.out_of_sample)["sharpe"],
            }
        )
pd.DataFrame(rows).round(3)

,panel,arch,best_epoch,val_IC,IS_sharpe,holdout_sharpe
0,default,mlp,2,0.015,0.756,0.153
1,default,cnn,1,0.006,-0.935,-0.938
2,default,lstm,1,0.013,-0.322,1.046
3,3x drift,mlp,3,0.059,3.581,-0.902
4,3x drift,cnn,9,0.134,4.215,0.094
5,3x drift,lstm,11,0.030,1.320,-0.436


## #10: explaining returns vs predicting them
Total R² measures how well the factors reconstruct returns; predictive R² measures whether beta(z) · (average factor) forecasts them.

In [3]:
cae = ConditionalAutoencoder()
run = run_strategy(cae, prices, train_end)
print({k: round(v, 3) for k, v in cae.r2_.items()})
print("factor premia (per 5 days):", cae.premia_.round(4))
print(
    "IS / hold-out Sharpe:",
    round(tear_sheet(run.in_sample)["sharpe"], 2),
    round(tear_sheet(run.out_of_sample)["sharpe"], 2),
)

{'total_train': 0.335, 'total_val': 0.332, 'predictive_val': -0.015}
factor premia (per 5 days): [-0.1921  0.0404  0.2406]
IS / hold-out Sharpe: 2.52 -0.53
